In [1]:
!git clone https://github.com/kushagrakushwah/saas-openenv-audit.git
%cd saas-openenv-audit

fatal: destination path 'saas-openenv-audit' already exists and is not an empty directory.
/content/saas-openenv-audit


In [2]:
import os
repo_path = '/content/saas-openenv-audit'

if os.path.exists(repo_path):
    %cd {repo_path}
    !pip install -r requirements.txt
else:
    print("Repo not found. Check the git clone step!")

/content/saas-openenv-audit


In [6]:
from huggingface_hub import notebook_login
notebook_login()

In [12]:
# 1. Clone the repository
!git clone https://github.com/kushagrakushwah/saas-openenv-audit.git

# 2. Move into the directory
%cd saas-openenv-audit

# 3. Switch to the 'kushagra' branch
!git checkout kushagra

# 4. Install Unsloth and Hackathon dependencies (H+09 Sprint)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

# 5. Verify the branch and files
!git branch
!ls server/

fatal: destination path 'saas-openenv-audit' already exists and is not an empty directory.
/content/saas-openenv-audit/saas-openenv-audit
Already on 'kushagra'
Your branch is up to date with 'origin/kushagra'.
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-37wgw5qg/unsloth_c17e44d3bce848aaab87ae13880c3629
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-37wgw5qg/unsloth_c17e44d3bce848aaab87ae13880c3629
  Resolved https://github.com/unslothai/unsloth.git to commit b09aa82a3ac9ac9794c497e4b8f747f77e52b162
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
* kushagra
  main
app.py	   environment.py  __init__.py	scenarios.py
client.py  graders.py	   models.py	schemas.py


In [4]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-fdct7tsc/unsloth_9f3affe4ae274c959d04a93e7bdad580
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-fdct7tsc/unsloth_9f3affe4ae274c959d04a93e7bdad580
  Resolved https://github.com/unslothai/unsloth.git to commit b09aa82a3ac9ac9794c497e4b8f747f77e52b162
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU: Tesla T4
VRAM: 15.6 GB


In [5]:
from unsloth import FastLanguageModel
import torch
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)
print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded. VRAM used: 7.19 GB


In [15]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# 1. Apply the chat template format
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

# 2. Format the data function
def format_trajectory(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": text}

# 3. Pull from your Hugging Face repo
dataset = load_dataset("kushagrakushwah/envaudit-sft-data", split = "train")
dataset = dataset.map(format_trajectory, remove_columns=["total_reward", "grader_score"])

print(f"Dataset loaded: {len(dataset)} trajectories ready.")

sft_data.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset loaded: 500 trajectories ready.


In [6]:
from unsloth import FastLanguageModel
import torch
import os

# Set sequence length to 128 for survival on T4
MAX_SEQ_LEN = 128

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 4, # Ghost Mode: minimal rank to save memory
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model loaded. VRAM used: 7.23 GB


In [16]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # The dataset is now loaded!
    dataset_text_field = "text",
    max_seq_length = 128,
    dataset_num_proc = 1,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = True,
        optim = "paged_adamw_8bit",
        logging_steps = 1,
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 10,092,544 of 7,625,709,056 (0.13% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
Unsloth: Input IDs of shape torch.Size([1, 1024]) with length 1024 > the model's max sequence length of 128.
We shall truncate it ourselves. It's imperative if you correct this issue first.


ValueError: Expected input batch_size (128) to match target batch_size (1024).

In [18]:
import torch, gc
try: del trainer, model
except: pass
gc.collect()
torch.cuda.empty_cache()

In [1]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [2]:
import os
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Force aggressive memory management
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# 2. 512 is the maximum safe sequence length for the T4 backward pass
MAX_LEN = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = MAX_LEN,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 4,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

def format_trajectory(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )}

dataset = load_dataset("kushagrakushwah/envaudit-sft-data", split="train")

# 3. CRITICAL FIX: load_from_cache_file=False forces a clean tokenization
# This entirely prevents the TorchDynamo tensor mismatch crash.
dataset = dataset.map(
    format_trajectory,
    remove_columns=["total_reward", "grader_score"],
    load_from_cache_file=False
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_LEN,
    dataset_num_proc = 1,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = True,
        optim = "paged_adamw_8bit",
        logging_steps = 1,
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",
    ),
)

print("\n--- LAUNCHING DEFINITIVE TRAINING ---")
trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



--- LAUNCHING DEFINITIVE TRAINING ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 10,092,544 of 7,625,709,056 (0.13% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
Unsloth: Input IDs of shape torch.Size([1, 1024]) with length 1024 > the model's max sequence length of 512.
We shall truncate it ourselves. It's imperative if you correct this issue first.


ValueError: Expected input batch_size (512) to match target batch_size (1024).

In [1]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [2]:
import os
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments

# Force aggressive memory management
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MAX_LEN = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = MAX_LEN,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 4,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

def format_trajectory(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = load_dataset("kushagrakushwah/envaudit-sft-data", split="train")

# Format to raw text
dataset = dataset.map(format_trajectory, remove_columns=["total_reward", "grader_score"], load_from_cache_file=False)

# --- THE BULLETPROOF FIX ---
# Tokenize the text, chop it exactly at 512 tokens, and decode it back to a string.
# The SFTTrainer will now never receive a sequence that causes a tensor mismatch.
def hard_truncate(example):
    tokens = tokenizer(example["text"], truncation=True, max_length=MAX_LEN, add_special_tokens=False)
    example["text"] = tokenizer.decode(tokens["input_ids"])
    return example

dataset = dataset.map(hard_truncate, load_from_cache_file=False)
# ---------------------------

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_LEN,
    dataset_num_proc = 1,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = True,
        optim = "paged_adamw_8bit",
        logging_steps = 1,
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",
    ),
)

print("\n--- LAUNCHING DEFINITIVE TRAINING ---")
trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



--- LAUNCHING DEFINITIVE TRAINING ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 10,092,544 of 7,625,709,056 (0.13% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.625205
2,2.625205
3,2.557444
4,2.364860
5,2.148213
6,1.946566
7,1.762741
8,1.604967
9,1.442881
10,1.282036


TrainOutput(global_step=60, training_loss=0.40861350641450067, metrics={'train_runtime': 1127.0318, 'train_samples_per_second': 0.852, 'train_steps_per_second': 0.053, 'total_flos': 2.062084507435008e+16, 'train_loss': 0.40861350641450067, 'epoch': 1.896})

In [3]:
# Save the adapters and tokenizer locally
model.save_pretrained("envaudit_sft_model")
tokenizer.save_pretrained("envaudit_sft_model")
print("Local save complete!")

Unsloth: Restored added_tokens_decoder metadata in envaudit_sft_model/tokenizer_config.json.


Local save complete!


In [4]:
# Push to the Hub so Raj can access it for the PPO scaffold
model.push_to_hub("kushagrakushwah/envaudit-qwen-7b-sft", token = True)
tokenizer.push_to_hub("kushagrakushwah/envaudit-qwen-7b-sft", token = True)
print("Push to Hub complete! Hand-off ready.")

README.md:   0%|          | 0.00/585 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.9kB / 40.4MB            

Saved model to https://huggingface.co/kushagrakushwah/envaudit-qwen-7b-sft


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpawnyvf9k/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpawnyvf9k/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

Push to Hub complete! Hand-off ready.
